[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chihuahualee828/TorchCode/blob/master/templates/42_mini_llm.ipynb)

# Final Assignment 42: Decoder-only LLM

Build a complete, trainable **Llama-style MiniLLM** using `LLMConfig`. Assignment 43 imports your model for training and chat. Design the implementation yourself.

## Requirements

- Token embeddings, exactly `config.num_layers` decoder layers, a final RMSNorm before the vocabulary head, and tied embedding/output weights.
- **Pre-norm:** within each layer, causal grouped-query attention comes first, then SwiGLU. Each sublayer normalizes its input with its own RMSNorm before computation and adds its output to the unnormalized residual stream. There is no post-residual norm within the layer.
- Use `nn.RMSNorm` with `config.eps` (PyTorch 2.4+), bias-free projections and no dropout.
- RoPE on Q/K: full head width, adjacent pairs, positions starting at zero, base `config.rope_base`, and frequency `base ** (-2*i/head_dim)` for pair `i`. Consecutive query-head groups share a KV head. No learned position embeddings.

**Allowed:** basic PyTorch layers/functions, normalization, activations, autograd and compatible built-in RoPE operations. A custom RoPE class/function is **not required**. Implement attention and decoder/model assembly yourself; no prebuilt attention/Transformer models, pretrained weights or judge oracle calls.

PyTorch documents [`torch.onnx.ops.rotary_embedding`](https://docs.pytorch.org/docs/2.14/onnx_ops.html#torch.onnx.ops.rotary_embedding), but its direct call fails backward in the tested environment. Any RoPE implementation you choose must preserve gradients and match the specified convention. The solution uses differentiable tensor operations.

## Configuration

Architecture choices above are fixed; dimensions below come from `LLMConfig`.
Use the passed configuration rather than hardcoding defaults: the judge varies them.

| Field | Meaning | Default |
|---|---|---|
| `num_layers` | Number of decoder layers | 2 |
| `d_model` | Embedding and residual width | 64 |
| `num_heads` | Query heads | 4 |
| `num_kv_heads` | Key/value heads | 2 |
| `hidden_dim` | SwiGLU intermediate width | 128 |
| `vocab_size` | Input/output vocabulary size | 260 |
| `max_seq_len` | Maximum input length | 256 |
| `rope_base` | RoPE frequency base | 10000.0 |
| `eps` | RMSNorm epsilon | 1e-5 |

`LLMConfig` validates positive dimensions, head divisibility and even head width.
Pre-norm, SwiGLU, RoPE and weight tying are requirements, not config switches.

## Interface and grading contract

`MiniLLM(config)` must be an `nn.Module`. `forward(input_ids)` takes integer IDs `(B,T)` and returns raw logits `(B,T,vocab_size)`. Reject non-2D inputs and lengths outside `1..max_seq_len` with `ValueError`. Preserve dtype, device and gradients; support right padding.

Component names are your choice. The checker discovers them from their module types and dimensions. Use one `ModuleList` with `config.num_layers` decoder layers; each layer needs separate bias-free Q/K/V/O and gate/up/down projections, plus two `nn.RMSNorm(d_model)` modules. The model needs one `nn.Embedding(vocab_size, d_model)`, one final `nn.RMSNorm(d_model)`, and one bias-free `nn.Linear(d_model, vocab_size)` head. Tie the head's `weight` to the embedding's `weight`. The feed-forward width is `config.hidden_dim`; attention head width is `d_model // num_heads`.

## Required KV cache

`forward(input_ids, past_key_values=None, use_cache=False)`: the default still returns logits for ordinary training and assignment 43. With `use_cache=True`, return `(logits, present_key_values)`. Pass only new token IDs when supplying a cache; logits cover only those new tokens.

- Cache: a tuple/list of `config.num_layers` `(K, V)` pairs in decoder execution order. Each tensor has shape `(B, num_kv_heads, cached_length, head_dim)`, matching model dtype/device. Store RoPE-rotated K and unrotated V before repeating KV heads.
- Prefill with `past_key_values=None`; append new K/V on later calls. RoPE positions continue from `cached_length`. New tokens can attend to all cached tokens and earlier/current tokens in their new chunk.
- Process only new tokens through embeddings and decoder projections. Do not reproject cached tokens, re-rotate cached K, mutate the supplied cache, or keep hidden cache state on the model. Independent conversations must remain independent.
- Raise `ValueError` if prefix length plus new length exceeds `max_seq_len`, a cache has invalid layer count/rank/batch/head dimensions or inconsistent lengths/dtype/device, or a cache is supplied with `use_cache=False`. Empty new-token inputs are invalid; zero-length cache tensors are allowed.

Cached inference is tested in eval/no-grad mode with unpadded batches of equal prefix length. Checks cover chunked and token-wise equivalence, compact GQA/MQA/MHA caches, incremental projection work, branch reuse, context limits and identical greedy token IDs. Choose your own internal cache/helper names; only this public interface is required.

## Submit

Run the implementation cell, reload the exported module, then run `check("mini_llm")`. Fixed CPU checks cover logits, parameter gradients, causality, head configurations, context limits and serialization. Results are reproducible in the same environment.

The exported `mini_llm.py` must be available to assignment 43.


In [ ]:
# Use this repository version: these new tasks may not be on PyPI yet.
# Local: install with `pip install -e /path/to/TorchCode` before launching Jupyter.
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q git+https://github.com/chihuahualee828/TorchCode.git@master')
except ImportError:
    pass

import torch
from torch_judge import check, hint
from torch_judge.capstone import LLMConfig, deterministic

if not hasattr(torch.nn, "RMSNorm"):
    raise ImportError("Assignment 42 requires PyTorch 2.4+. Install torch>=2.4 and restart the kernel.")


In [ ]:
%%writefile mini_llm.py
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch_judge.capstone import LLMConfig


class DecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        raise NotImplementedError("Implement __init__")

    def forward(self, x):
        raise NotImplementedError("Implement forward")


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config


    def forward(self, input_ids):
        raise NotImplementedError("Implement forward")


In [ ]:
import importlib
import mini_llm
importlib.reload(mini_llm)
MiniLLM = mini_llm.MiniLLM


In [ ]:
with deterministic(0):
    model = MiniLLM(LLMConfig())
    print(model(torch.tensor([[1, 12, 25, 2]])).shape)
    print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
check('mini_llm')